<div style="background-color: #ffcccc; padding: 10px; border: 1px solid red;">
    <h3><strong>Remark</strong></h3>
    The data queried in this notebook is not used for the isoforms database (data was directly downloaded from UniProtKB, see notebook 1.2)
</div>

# Get full human proteome and ioforms from UniProt
In this Notebook we will use SPARQL to query UniProt RDF data and get a list of Accession IDs and UniProt Rest API to fetch the isoform data since it is more robust for large lists of proteins than SPARQL.

# Setup
## Import Packages
SPARQLWrapper is a simple Python wrapper around a SPARQL service to remotelly execute your queries. It helps increating the queryinvokationand,possibly, convertthe result into a more manageable format.

In [ ]:
import sys
import os
import session_info

In [ ]:
import pandas as pd
from SPARQLWrapper import SPARQLWrapper, JSON
import time
import requests

In [ ]:
# Display session information
session_info.show()

## Set working directory

In [ ]:
cwd = os.getcwd()
if not cwd.endswith("Isoform_Database/Jupyter_environment/01_UniProt"):
    print("WARNING: The working directory is not set to the '01_UniProt'!")
    print("Current working directory:", cwd)
else:
    print(f"Working directory is correctly set to the '01_Uniprot' folder (\"{cwd}\").")

In [ ]:
%%bash -s "data"
mkdir -p "$1"

In [ ]:
# Data directory for fetched data
data_dir = "data"
raw_data_dir = "data/raw_data"

# Querying UniProt
## Get canonical forms of all human proteins in UniProt
The next cell is skipped by default. If you run this notebook for the first time change the next cell from raw to code in order to run it.

In [ ]:
full_proteome = pd.read_csv("data/human_proteome_list.csv")

In [ ]:
#check how many proteins were fetched by query
print(f"Found {len(full_proteome)} proteins in UniProt.")

#check how many unique protein ids were found
unique = len(full_proteome["ID"].unique())
print(f"There are {unique} unique IDs.")

In [ ]:
#take a quick look at the output table
full_proteome.head(10)

## Get protein isoforms
The following function fetches isoform sequences for a list of UniProt Accessions. For this task we are using the UniProt REST API because since this task is to computationally heavy for the UniProt SPARQL endpoint.

In [ ]:
#list of protein ids
accessions_list = full_proteome["ID"].unique().tolist()

In [ ]:
#take a quick look at the accessions list
accessions_list[0:10]

In [ ]:
#read in query output table
full_proteome_isoforms = pd.read_csv("data/human_proteome_isoforms_list.csv")

In [ ]:
#check how many proteins were fetched by query
print(f"Found {len(full_proteome_isoforms)} isoforms in UniProt.")

#check how many unique protein ids were found
len(full_proteome_isoforms["isoform_id"].unique())

In [ ]:
#take a quick look at the output table
full_proteome_isoforms.head(10)

## Check if IDs and sequences match between the two dataframes

In [ ]:
#check if all the canonical protein IDs are in the isoform df
full_proteome["ID"].isin(full_proteome_isoforms["ID"]).all()

In [ ]:
#check if all the canonical protein sequences are in the isoform df
full_proteome["Sequence"].isin(full_proteome_isoforms["sequence"]).all()

In [ ]:
#check how many are a match
mask = full_proteome["Sequence"].isin(full_proteome_isoforms["sequence"])

print(f"{len(full_proteome[mask])} canonical records have a sequence found in the isoform df.")
print(f"{len(full_proteome[~mask])} canonical records have no matching sequence in the isoform df.")

In [ ]:
#have a look at the ones that don't match
no_match_seq = full_proteome[~mask].reset_index(drop=True)
no_match_seq

In [ ]:
#other way around
#check how many of the isoform protein IDs are in the canonical df
full_proteome_isoforms["ID"].isin(full_proteome["ID"]).sum()

In [ ]:
#check how many of the isoform protein sequences are in the canonical df
full_proteome_isoforms["sequence"].isin(full_proteome["Sequence"]).sum()

In [ ]:
len(full_proteome["Sequence"].unique())

In [ ]:
len(full_proteome_isoforms["sequence"].unique())